# Original BERT SMS Scam Classifier

This notebook fine-tunes `bert-base-uncased` on `datasets/SMSSpamCollection_train_subset` and evaluates on `datasets/SMSSpamCollection_ablation_test`. It keeps the same training hyperparameters used in the SingBERT classifier notebook.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from IPython.display import display
from sklearn.metrics import accuracy_score, classification_report, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)


In [ ]:
# Project paths and training config
PROJECT_ROOT = Path.cwd()
TRAIN_DATA_PATH = PROJECT_ROOT / "datasets" / "SMSSpamCollection_train_subset"
TEST_DATA_PATH = PROJECT_ROOT / "datasets" / "SMSSpamCollection_ablation_test"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
MODEL_OUTPUT_DIR = ARTIFACT_DIR / "original_bert_classifier"
METRICS_OUTPUT_DIR = ARTIFACT_DIR / "original_bert_metrics"

MODEL_CHECKPOINT = "bert-base-uncased"

RANDOM_SEED = 42
VALIDATION_SIZE = 0.15
MAX_LENGTH = 128
NUM_EPOCHS = 4
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
WEIGHT_DECAY = 0.01

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

set_seed(RANDOM_SEED)
pd.set_option("display.max_colwidth", 160)

for data_path in [TRAIN_DATA_PATH, TEST_DATA_PATH]:
    if not data_path.exists():
        raise FileNotFoundError(f"Missing dataset file: {data_path}")

print(f"Training dataset: {TRAIN_DATA_PATH}")
print(f"Ablation test dataset: {TEST_DATA_PATH}")
print(f"Model checkpoint: {MODEL_CHECKPOINT}")


In [ ]:
# Load the train and test files and map labels to scam / not_scam
def load_sms_dataframe(data_path):
    rows = []
    with data_path.open("r", encoding="utf-8", errors="replace") as fp:
        for line_number, raw_line in enumerate(fp, start=1):
            line = raw_line.rstrip("\n")
            if not line:
                continue
            if "\t" not in line:
                raise ValueError(f"Line {line_number} is missing a tab separator: {line!r}")
            raw_label, text = line.split("\t", 1)
            rows.append({"raw_label": raw_label.strip(), "text": text.strip()})
    return pd.DataFrame(rows)

train_source_df = load_sms_dataframe(TRAIN_DATA_PATH)
test_df = load_sms_dataframe(TEST_DATA_PATH)

label_name_map = {"ham": "not_scam", "spam": "scam"}
for df in [train_source_df, test_df]:
    df["label"] = df["raw_label"].map(label_name_map)

if train_source_df["label"].isna().any() or test_df["label"].isna().any():
    unknown_labels = sorted(
        set(train_source_df.loc[train_source_df["label"].isna(), "raw_label"].unique())
        | set(test_df.loc[test_df["label"].isna(), "raw_label"].unique())
    )
    raise ValueError(f"Found unknown labels: {unknown_labels}")

for df in [train_source_df, test_df]:
    df["text"] = df["text"].astype(str).str.strip()
    df["label"] = df["label"].astype(str).str.strip()
    df.drop_duplicates(subset=["text", "label"], inplace=True)
    df.reset_index(drop=True, inplace=True)
label2id = {"not_scam": 0, "scam": 1}
id2label = {idx: label for label, idx in label2id.items()}
train_source_df["labels"] = train_source_df["label"].map(label2id)
test_df["labels"] = test_df["label"].map(label2id)

display(train_source_df.head())
print("Training source counts:")
print(train_source_df["label"].value_counts().sort_index().to_string())
print()
print("Held-out ablation test counts:")
print(test_df["label"].value_counts().sort_index().to_string())


In [ ]:
# Split the training subset into train and validation, and keep the ablation set fixed as test
train_df, validation_df = train_test_split(
    train_source_df,
    test_size=VALIDATION_SIZE,
    random_state=RANDOM_SEED,
    stratify=train_source_df["labels"],
)

split_frames = {"train": train_df, "validation": validation_df, "test": test_df}
for split_name, split_df in split_frames.items():
    print(f"{split_name:>10}: {len(split_df):4d} rows")
    print(split_df["label"].value_counts().sort_index().to_string())
    print()

dataset_dict = DatasetDict(
    {
        split_name: Dataset.from_pandas(split_df.reset_index(drop=True), preserve_index=False)
        for split_name, split_df in split_frames.items()
    }
)


In [ ]:
# Tokenize and prepare metrics
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, use_fast=True)

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

tokenized_datasets = dataset_dict.map(tokenize_batch, batched=True)

tensor_columns = ["input_ids", "attention_mask", "labels"]
if "token_type_ids" in tokenized_datasets["train"].column_names:
    tensor_columns.append("token_type_ids")

tokenized_datasets.set_format(type="torch", columns=tensor_columns)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0,
    )
    precision_m, recall_m, f1_m, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0,
    )
    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision_weighted": precision_w,
        "recall_weighted": recall_w,
        "f1_weighted": f1_w,
        "precision_macro": precision_m,
        "recall_macro": recall_m,
        "f1_macro": f1_m,
    }

label_mapping_path = METRICS_OUTPUT_DIR / "label_mapping.json"
with label_mapping_path.open("w", encoding="utf-8") as fp:
    json.dump(
        {
            "label2id": label2id,
            "id2label": {str(k): v for k, v in id2label.items()},
        },
        fp,
        indent=2,
    )

print(f"Saved label mapping to: {label_mapping_path}")


In [ ]:
# Fine-tune the original BERT classifier
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(label2id),
    label2id=label2id,
    id2label=id2label,
)

training_args_kwargs = dict(
    output_dir=str(MODEL_OUTPUT_DIR),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    save_total_limit=2,
    seed=RANDOM_SEED,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

if "eval_strategy" in TrainingArguments.__dataclass_fields__:
    training_args_kwargs["eval_strategy"] = "epoch"
else:
    training_args_kwargs["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**training_args_kwargs)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
trainer.save_model(str(MODEL_OUTPUT_DIR))
tokenizer.save_pretrained(str(MODEL_OUTPUT_DIR))

print(f"Saved fine-tuned model to: {MODEL_OUTPUT_DIR}")


In [ ]:
# Evaluate and save metrics
def to_python_scalars(value):
    if isinstance(value, dict):
        return {k: to_python_scalars(v) for k, v in value.items()}
    if isinstance(value, list):
        return [to_python_scalars(v) for v in value]
    if isinstance(value, tuple):
        return [to_python_scalars(v) for v in value]
    if isinstance(value, np.generic):
        return value.item()
    return value

def normalize_metric_keys(metrics, prefix):
    normalized = {}
    prefix_text = f"{prefix}_"
    for key, value in metrics.items():
        if key.startswith(prefix_text):
            normalized[key[len(prefix_text):]] = value
        else:
            normalized[key] = value
    return normalized

def evaluate_split(split_name, dataset_split):
    prediction_output = trainer.predict(dataset_split, metric_key_prefix=split_name)
    metrics = normalize_metric_keys(prediction_output.metrics, split_name)
    predicted_ids = np.argmax(prediction_output.predictions, axis=-1)
    report = classification_report(
        prediction_output.label_ids,
        predicted_ids,
        target_names=[id2label[idx] for idx in sorted(id2label)],
        digits=4,
        zero_division=0,
        output_dict=True,
    )
    report_df = pd.DataFrame(report).transpose()
    report_path = METRICS_OUTPUT_DIR / f"{split_name}_classification_report.csv"
    report_df.to_csv(report_path)
    return to_python_scalars(metrics), report_df

train_metrics, train_report_df = evaluate_split("train", tokenized_datasets["train"])
validation_metrics, validation_report_df = evaluate_split("validation", tokenized_datasets["validation"])
test_metrics, test_report_df = evaluate_split("test", tokenized_datasets["test"])

summary_metrics = {
    "model_checkpoint": MODEL_CHECKPOINT,
    "train_data_path": str(TRAIN_DATA_PATH),
    "test_data_path": str(TEST_DATA_PATH),
    "output_model_dir": str(MODEL_OUTPUT_DIR),
    "split_sizes": {split_name: len(split_df) for split_name, split_df in split_frames.items()},
    "best_model_checkpoint": trainer.state.best_model_checkpoint,
    "train_runtime": to_python_scalars(train_result.metrics),
    "train": train_metrics,
    "validation": validation_metrics,
    "test": test_metrics,
}

summary_metrics_path = METRICS_OUTPUT_DIR / "summary_metrics.json"
with summary_metrics_path.open("w", encoding="utf-8") as fp:
    json.dump(summary_metrics, fp, indent=2)

trainer_log_path = METRICS_OUTPUT_DIR / "trainer_log_history.json"
with trainer_log_path.open("w", encoding="utf-8") as fp:
    json.dump([to_python_scalars(item) for item in trainer.state.log_history], fp, indent=2)

metrics_overview = pd.DataFrame([train_metrics, validation_metrics, test_metrics], index=["train", "validation", "test"])
display(metrics_overview)
display(test_report_df)

print(f"Saved metric summary to: {summary_metrics_path}")
print(f"Saved trainer history to: {trainer_log_path}")
print(f"Saved per-class reports to: {METRICS_OUTPUT_DIR}")
